# Question 3 — Conditional GAN (pix2pix) for Sketch-to-Face Generation

**Objective:** Implement a Conditional GAN that takes a face **sketch** as the conditioning
input and generates the corresponding **realistic face photo**, trained on the paired
*Person Face Sketches* dataset.

This notebook implements the image-conditional GAN described by Isola et al.,
*"Image-to-Image Translation with Conditional Adversarial Networks" (pix2pix, CVPR 2017)*,
which is the standard way of applying the original Conditional GAN idea (Mirza & Osindero, 2014)
to a full-image condition (sketch -> photo) rather than a class-label condition.

**Architecture**
- **Generator:** U-Net encoder-decoder with skip connections (sketch -> RGB face).
- **Discriminator:** Conditional PatchGAN. It receives the *(sketch, photo)* pair
  concatenated channel-wise and outputs a grid of real/fake scores instead of a single
  scalar, which encourages sharp, locally-realistic textures.
- **Loss:** Adversarial loss (BCE-with-logits) + weighted L1 reconstruction loss,
  exactly as in the pix2pix objective `L = L_GAN + lambda * L_L1`.

Run this notebook top-to-bottom on Google Colab (GPU runtime recommended). Checkpoints are
saved after every epoch so training can be resumed if the runtime disconnects.


## 1. Setup

In [ ]:
# Uncomment on a fresh Colab runtime
# !pip install -q kagglehub torch torchvision matplotlib


In [ ]:
import os
import math
import random
import glob
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.utils import make_grid, save_image

from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True  # autotune conv algorithms for our fixed input size
print("Using device:", DEVICE)


## 2. Dataset — Person Face Sketches

The dataset pairs a hand-drawn/generated **sketch** with the corresponding **real face
photo**. Download it from Kaggle (`almightyj/person-face-sketches`) or point `DATA_DIR`
at a local copy. The loader below walks the dataset directory, groups files into
"sketch" vs "photo" using the folder/file name, and pairs them by matching filename stem
— this works regardless of whether the dataset ships with `train/val` subfolders.


In [ ]:
def find_image_files(root):
    """Recursively collect all image file paths under root."""
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    files = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if os.path.splitext(f)[1].lower() in exts:
                files.append(os.path.join(dirpath, f))
    return files


def locate_dataset():
    """Find a local copy of the Person Face Sketches dataset.

    Tries, in order: a Kaggle Notebook's auto-mounted "Add Data" input, a kagglehub
    download (requires a Kaggle API token), and a couple of common manual-extraction
    paths. Returns the first candidate directory that actually contains images, so a
    silently-failed download doesn\'t turn into a confusing "0 pairs found" later on.
    """
    candidates = []
    candidates += sorted(glob.glob("/kaggle/input/*sketch*"))
    candidates += sorted(glob.glob("/kaggle/input/*face*"))

    try:
        import kagglehub
        candidates.append(kagglehub.dataset_download("almightyj/person-face-sketches"))
    except Exception as e:
        print(f"kagglehub download unavailable: {e}")

    candidates += ["data/person-face-sketches", "/content/person-face-sketches"]

    print("Searching candidate dataset locations:")
    for c in candidates:
        if c and os.path.isdir(c):
            n_files = len(find_image_files(c))
            print(f"  {c}  ->  {n_files} image files")
            if n_files > 0:
                return c
        else:
            print(f"  {c}  ->  not found")
    return None


DATA_DIR = locate_dataset()
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate the Person Face Sketches dataset automatically. If you\'re "
        "on Kaggle, click \'Add Data\' and attach the dataset; on Colab, download/unzip "
        "it and set DATA_DIR manually to the folder containing the images."
    )
print("Using DATA_DIR:", DATA_DIR)


In [ ]:
def build_pairs(root_dir):
    """Pair sketch/photo images by matching filename stem within each dataset split.

    Classification is based on the *immediate parent directory name* of each file
    (e.g. ".../train/sketches/123.jpg" -> domain "sketches", split "train"), not the
    full path — matching on the full path is a trap here, since the dataset root
    itself is called "person-face-sketches" and would make every file look like it
    belongs to the sketch domain. The split (train/val) is taken from the dataset\'s
    own directory layout rather than re-split randomly, so results are reproducible
    and match how the dataset was intended to be evaluated.
    """
    all_files = find_image_files(root_dir)
    sketches, photos = {}, {}
    for f in all_files:
        domain_dir = os.path.basename(os.path.dirname(f)).lower()
        split_dir = os.path.basename(os.path.dirname(os.path.dirname(f))).lower()
        stem = os.path.splitext(os.path.basename(f))[0]
        split = split_dir if split_dir in ("train", "val") else "train"
        key = (split, stem)
        if "sketch" in domain_dir:
            sketches[key] = f
        elif "photo" in domain_dir:
            photos[key] = f
    common = sorted(set(sketches) & set(photos))
    train_pairs = [(sketches[k], photos[k]) for k in common if k[0] == "train"]
    val_pairs = [(sketches[k], photos[k]) for k in common if k[0] == "val"]
    return train_pairs, val_pairs


train_pairs, val_pairs = build_pairs(DATA_DIR)
all_files = find_image_files(DATA_DIR)
print(f"Found {len(all_files)} total images under DATA_DIR")
print(f"Train pairs: {len(train_pairs)} | Val pairs: {len(val_pairs)} (full dataset)")

# Subsample to keep each epoch fast on a single Kaggle/Colab GPU session. Increase or
# remove these caps if you have more time budget and want to train on the full set.
MAX_TRAIN_PAIRS = 6000
MAX_VAL_PAIRS = 1000
if len(train_pairs) > MAX_TRAIN_PAIRS:
    train_pairs = random.sample(train_pairs, MAX_TRAIN_PAIRS)
if len(val_pairs) > MAX_VAL_PAIRS:
    val_pairs = random.sample(val_pairs, MAX_VAL_PAIRS)
print(f"Using {len(train_pairs)} train pairs | {len(val_pairs)} val pairs (subsampled)")

if len(train_pairs) == 0:
    print("\nNo training pairs matched. Sample of files found (parent dir -> path):")
    for f in all_files[:15]:
        print(f"  [{os.path.basename(os.path.dirname(f))}] {f}")
    raise AssertionError(
        "No paired sketch/photo files found. Inspect the sample paths above: adjust "
        "build_pairs() if the sketch/photo folders aren\'t named \'sketches\'/\'photos\'."
    )


In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 32
NUM_WORKERS = 4

class SketchFaceDataset(Dataset):
    """Loads matched (sketch, photo) pairs and applies identical random-crop /
    horizontal-flip augmentation to both images of a pair so alignment is preserved."""

    def __init__(self, pairs, img_size=IMG_SIZE, augment=True):
        self.pairs = pairs
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        sketch_path, photo_path = self.pairs[idx]
        sketch = Image.open(sketch_path).convert("RGB")
        photo = Image.open(photo_path).convert("RGB")

        resize_dim = int(self.img_size * 1.12)
        sketch = TF.resize(sketch, [resize_dim, resize_dim], Image.BICUBIC)
        photo = TF.resize(photo, [resize_dim, resize_dim], Image.BICUBIC)

        if self.augment:
            i, j, h, w = transforms.RandomCrop.get_params(
                sketch, output_size=(self.img_size, self.img_size)
            )
            sketch = TF.crop(sketch, i, j, h, w)
            photo = TF.crop(photo, i, j, h, w)
            if random.random() > 0.5:
                sketch = TF.hflip(sketch)
                photo = TF.hflip(photo)
        else:
            sketch = TF.center_crop(sketch, [self.img_size, self.img_size])
            photo = TF.center_crop(photo, [self.img_size, self.img_size])

        sketch = TF.normalize(TF.to_tensor(sketch), [0.5] * 3, [0.5] * 3)
        photo = TF.normalize(TF.to_tensor(photo), [0.5] * 3, [0.5] * 3)
        return sketch, photo


train_set = SketchFaceDataset(train_pairs, augment=True)
val_set = SketchFaceDataset(val_pairs, augment=False)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True,
                           pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=NUM_WORKERS,
                         pin_memory=True, persistent_workers=True)

print(f"Train samples: {len(train_set)} | Val samples: {len(val_set)}")


In [ ]:
def denorm(t):
    return (t * 0.5 + 0.5).clamp(0, 1)


sketch_b, photo_b = next(iter(train_loader))
grid = make_grid(torch.cat([denorm(sketch_b[:4]), denorm(photo_b[:4])]), nrow=4)
plt.figure(figsize=(10, 5))
plt.axis("off")
plt.title("Top: sketches (condition)  |  Bottom: target photos")
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.show()


## 3. Model

### Generator — U-Net
An 8-level encoder/decoder with skip connections between mirrored layers, so fine
spatial detail from the sketch (edges, proportions) is passed directly to the decoder
instead of having to be reconstructed from the bottleneck.

### Discriminator — Conditional PatchGAN
Takes the sketch and a photo (real or generated) concatenated on the channel axis, and
classifies overlapping 70x70 patches as real/fake rather than the whole image, which is
what gives pix2pix its sharp local texture.


In [ ]:
class UNetDown(nn.Module):
    def __init__(self, in_c, out_c, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_c))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class UNetUp(nn.Module):
    def __init__(self, in_c, out_c, dropout=0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_c),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = self.model(x)
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, base=64):
        super().__init__()
        self.down1 = UNetDown(in_channels, base, normalize=False)
        self.down2 = UNetDown(base, base * 2)
        self.down3 = UNetDown(base * 2, base * 4)
        self.down4 = UNetDown(base * 4, base * 8, dropout=0.5)
        self.down5 = UNetDown(base * 8, base * 8, dropout=0.5)
        self.down6 = UNetDown(base * 8, base * 8, dropout=0.5)
        self.down7 = UNetDown(base * 8, base * 8, dropout=0.5)
        self.down8 = UNetDown(base * 8, base * 8, normalize=False, dropout=0.5)

        self.up1 = UNetUp(base * 8, base * 8, dropout=0.5)
        self.up2 = UNetUp(base * 16, base * 8, dropout=0.5)
        self.up3 = UNetUp(base * 16, base * 8, dropout=0.5)
        self.up4 = UNetUp(base * 16, base * 8, dropout=0.5)
        self.up5 = UNetUp(base * 16, base * 4)
        self.up6 = UNetUp(base * 8, base * 2)
        self.up7 = UNetUp(base * 4, base)

        self.final = nn.Sequential(
            nn.ConvTranspose2d(base * 2, out_channels, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        d8 = self.down8(d7)

        u1 = self.up1(d8, d7)
        u2 = self.up2(u1, d6)
        u3 = self.up3(u2, d5)
        u4 = self.up4(u3, d4)
        u5 = self.up5(u4, d3)
        u6 = self.up6(u5, d2)
        u7 = self.up7(u6, d1)
        return self.final(u7)


class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=6, base=64):
        super().__init__()

        def block(in_c, out_c, normalize=True, stride=2):
            layers = [nn.Conv2d(in_c, out_c, 4, stride, 1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(in_channels, base, normalize=False),
            *block(base, base * 2),
            *block(base * 2, base * 4),
            *block(base * 4, base * 8, stride=1),
            nn.Conv2d(base * 8, 1, 4, 1, 1),
        )

    def forward(self, sketch, photo):
        return self.model(torch.cat([sketch, photo], dim=1))


def weights_init(m):
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "InstanceNorm2d" in classname and getattr(m, "weight", None) is not None:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


In [ ]:
G = UNetGenerator().to(DEVICE)
D = PatchGANDiscriminator().to(DEVICE)
G.apply(weights_init)
D.apply(weights_init)

n_params_g = sum(p.numel() for p in G.parameters())
n_params_d = sum(p.numel() for p in D.parameters())
print(f"Generator params:     {n_params_g:,}")
print(f"Discriminator params: {n_params_d:,}")


## 4. Training

Objective (pix2pix): `L_G = L_GAN(G, D) + lambda_L1 * L_L1(G)`, with the discriminator
trained to maximise its ability to tell real `(sketch, photo)` pairs from generated
`(sketch, G(sketch))` pairs. `lambda_L1 = 100` follows the original paper and keeps the
generator's output close to the ground-truth face while the adversarial term supplies
high-frequency realism.

`latest.pt` is overwritten after every epoch (so training can always resume after a
Colab/Kaggle disconnect with at most one epoch of lost progress), but full checkpoints
are large — G + D + both Adam optimizer states runs a few hundred MB — so a separate,
uniquely-named rollback checkpoint is only kept every `MILESTONE_EVERY` epochs instead
of every epoch, to avoid exhausting the notebook's disk quota over a long run. Training
also runs under automatic mixed precision (AMP) when a GPU is available — it roughly
halves per-epoch time on Tensor Core GPUs (T4/P100/V100) with no accuracy cost, which
matters since a full 100-epoch run at 256x256 can otherwise exceed a single Kaggle GPU
session.


In [ ]:
NUM_EPOCHS = 100
LR = 2e-4
BETA1, BETA2 = 0.5, 0.999
LAMBDA_L1 = 100.0

CHECKPOINT_DIR = "checkpoints"
SAMPLE_DIR = "samples"
MILESTONE_EVERY = 20  # keep a full rollback checkpoint every N epochs, not every epoch
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA1, BETA2))
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(BETA1, BETA2))
criterion_gan = nn.BCEWithLogitsLoss()
criterion_l1 = nn.L1Loss()

USE_AMP = DEVICE == "cuda"
scaler_G = GradScaler(device="cuda", enabled=USE_AMP)
scaler_D = GradScaler(device="cuda", enabled=USE_AMP)
print("Mixed precision (AMP):", "enabled" if USE_AMP else "disabled (no GPU)")


In [ ]:
def save_checkpoint(epoch, path):
    """Write atomically (save to a temp file, then rename) so a crash or a full disk
    mid-write can\'t leave a truncated, unloadable checkpoint at `path`.
    """
    tmp_path = path + ".tmp"
    torch.save({
        "epoch": epoch,
        "G_state": G.state_dict(),
        "D_state": D.state_dict(),
        "optG_state": opt_G.state_dict(),
        "optD_state": opt_D.state_dict(),
        "scalerG_state": scaler_G.state_dict(),
        "scalerD_state": scaler_D.state_dict(),
    }, tmp_path)
    os.replace(tmp_path, path)


def load_checkpoint(path):
    ckpt = torch.load(path, map_location=DEVICE)
    G.load_state_dict(ckpt["G_state"])
    D.load_state_dict(ckpt["D_state"])
    opt_G.load_state_dict(ckpt["optG_state"])
    opt_D.load_state_dict(ckpt["optD_state"])
    if "scalerG_state" in ckpt:
        scaler_G.load_state_dict(ckpt["scalerG_state"])
        scaler_D.load_state_dict(ckpt["scalerD_state"])
    return ckpt["epoch"]


def save_sample_grid(epoch, n=4):
    G.eval()
    sketch, photo = next(iter(val_loader))
    sketch, photo = sketch[:n].to(DEVICE), photo[:n].to(DEVICE)
    with torch.no_grad():
        fake = G(sketch)
    grid = make_grid(torch.cat([denorm(sketch), denorm(fake), denorm(photo)]), nrow=n)
    save_image(grid, os.path.join(SAMPLE_DIR, f"epoch_{epoch + 1:03d}.png"))
    G.train()
    return grid


In [ ]:
# Resume automatically if a checkpoint from a previous run exists
start_epoch = 0
latest_ckpt = os.path.join(CHECKPOINT_DIR, "latest.pt")
if os.path.exists(latest_ckpt):
    start_epoch = load_checkpoint(latest_ckpt) + 1
    print(f"Resumed from epoch {start_epoch}")

history = {"G_loss": [], "D_loss": [], "epoch_seconds": []}

for epoch in range(start_epoch, NUM_EPOCHS):
    G.train()
    D.train()
    g_losses, d_losses = [], []
    epoch_start = time.time()

    for sketch, photo in train_loader:
        sketch, photo = sketch.to(DEVICE), photo.to(DEVICE)

        # --- Train Discriminator: maximise real/fake separation ---
        with autocast(device_type="cuda", enabled=USE_AMP):
            fake_photo = G(sketch)
            pred_real = D(sketch, photo)
            pred_fake = D(sketch, fake_photo.detach())
            loss_d = 0.5 * (
                criterion_gan(pred_real, torch.ones_like(pred_real))
                + criterion_gan(pred_fake, torch.zeros_like(pred_fake))
            )
        opt_D.zero_grad()
        scaler_D.scale(loss_d).backward()
        scaler_D.step(opt_D)
        scaler_D.update()

        # --- Train Generator: fool D + match ground truth (L1) ---
        with autocast(device_type="cuda", enabled=USE_AMP):
            pred_fake_for_g = D(sketch, fake_photo)
            loss_g_gan = criterion_gan(pred_fake_for_g, torch.ones_like(pred_fake_for_g))
            loss_g_l1 = criterion_l1(fake_photo, photo) * LAMBDA_L1
            loss_g = loss_g_gan + loss_g_l1

        opt_G.zero_grad()
        scaler_G.scale(loss_g).backward()
        scaler_G.step(opt_G)
        scaler_G.update()

        g_losses.append(loss_g.item())
        d_losses.append(loss_d.item())

    epoch_seconds = time.time() - epoch_start
    g_avg = sum(g_losses) / len(g_losses)
    d_avg = sum(d_losses) / len(d_losses)
    history["G_loss"].append(g_avg)
    history["D_loss"].append(d_avg)
    history["epoch_seconds"].append(epoch_seconds)

    remaining_min = (NUM_EPOCHS - epoch - 1) * epoch_seconds / 60
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS} | G_loss: {g_avg:.4f} | D_loss: {d_avg:.4f} "
          f"| {epoch_seconds:.1f}s/epoch | est. remaining: {remaining_min:.1f} min")

    save_checkpoint(epoch, latest_ckpt)
    # Full checkpoints (G + D + both optimizers) are large — a few hundred MB each —
    # so only keep periodic rollback points instead of one per epoch, which would
    # otherwise exhaust Kaggle/Colab's disk quota well before training finishes.
    if (epoch + 1) % MILESTONE_EVERY == 0 or (epoch + 1) == NUM_EPOCHS:
        save_checkpoint(epoch, os.path.join(CHECKPOINT_DIR, f"epoch_{epoch + 1:03d}.pt"))
    save_sample_grid(epoch)


## 5. Results — training curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["G_loss"], label="Generator loss")
plt.plot(history["D_loss"], label="Discriminator loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Conditional GAN training loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

avg_epoch_s = sum(history["epoch_seconds"]) / len(history["epoch_seconds"])
print(f"Average epoch time: {avg_epoch_s:.1f}s ({avg_epoch_s / 60:.2f} min)")


### Qualitative results — sketch / generated / ground-truth triptychs

In [ ]:
grid = save_sample_grid(epoch=NUM_EPOCHS - 1, n=4)
plt.figure(figsize=(12, 6))
plt.axis("off")
plt.title("Row 1: input sketch | Row 2: generated face | Row 3: ground-truth photo")
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.show()


## 6. Quantitative evaluation on the validation split

L1 measures pixel-level reconstruction error and PSNR measures overall fidelity — neither
fully captures perceptual realism (that is what the adversarial loss is for), but together
with the qualitative grids above they give a sense of how well the model has converged.


In [ ]:
def evaluate(loader):
    G.eval()
    l1_total, psnr_total, n = 0.0, 0.0, 0
    with torch.no_grad():
        for sketch, photo in loader:
            sketch, photo = sketch.to(DEVICE), photo.to(DEVICE)
            fake = G(sketch)
            l1 = criterion_l1(fake, photo).item()
            mse = F.mse_loss(denorm(fake), denorm(photo)).item()
            psnr = 10 * math.log10(1.0 / max(mse, 1e-8))
            bs = sketch.size(0)
            l1_total += l1 * bs
            psnr_total += psnr * bs
            n += bs
    G.train()
    print(f"Validation L1: {l1_total / n:.4f}")
    print(f"Validation PSNR: {psnr_total / n:.2f} dB")


evaluate(val_loader)


## 7. Inference on a user-supplied sketch

Given any sketch image path, this generates the corresponding realistic face and
optionally saves it to disk. This is the function a UI/API layer would call.


In [ ]:
def generate_face_from_sketch(sketch_path, save_path=None, img_size=IMG_SIZE):
    G.eval()
    sketch = Image.open(sketch_path).convert("RGB")
    sketch = TF.resize(sketch, [img_size, img_size], Image.BICUBIC)
    tensor = TF.normalize(TF.to_tensor(sketch), [0.5] * 3, [0.5] * 3).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        fake = G(tensor)
    fake_img = TF.to_pil_image(denorm(fake.squeeze(0).cpu()))
    if save_path:
        fake_img.save(save_path)
    return fake_img


# Example:
# result = generate_face_from_sketch(val_pairs[0][0], save_path="generated_face.png")
# plt.imshow(result); plt.axis("off"); plt.show()


## 8. Save final model weights

The generator is the only network needed at inference time (e.g. behind a Flask API),
so it is saved separately alongside the full training checkpoint.


In [ ]:
torch.save(G.state_dict(), os.path.join(CHECKPOINT_DIR, "generator_final.pt"))
print("Saved final generator weights to", os.path.join(CHECKPOINT_DIR, "generator_final.pt"))


## 9. Bundle results for review

Zips the per-epoch sample grids plus the loss/timing history into a small archive —
handy for downloading and sharing for a results review without dragging along the
multi-hundred-MB checkpoint files.


In [ ]:
import json
import zipfile
from pathlib import Path

with open("training_history.json", "w") as f:
    json.dump(history, f, indent=2)

summary = {
    "num_epochs_run": len(history["G_loss"]),
    "final_G_loss": history["G_loss"][-1] if history["G_loss"] else None,
    "final_D_loss": history["D_loss"][-1] if history["D_loss"] else None,
    "avg_epoch_seconds": (
        sum(history["epoch_seconds"]) / len(history["epoch_seconds"])
        if history["epoch_seconds"] else None
    ),
    "train_pairs": len(train_pairs),
    "val_pairs": len(val_pairs),
}
with open("run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

output_zip = "results_bundle.zip"
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    samples_dir = Path(SAMPLE_DIR)
    if samples_dir.is_dir():
        for f in sorted(samples_dir.glob("*.png")):
            zf.write(f, f"samples/{f.name}")
    zf.write("training_history.json")
    zf.write("run_summary.json")

size_mb = Path(output_zip).stat().st_size / 1e6
print(f"Wrote {output_zip} ({size_mb:.1f} MB)")


In [ ]:
# Kaggle/Jupyter does not allow a script to force a download with zero user
# interaction (browsers block that for security), but this renders a one-click
# download link directly in the cell output below.
from IPython.display import FileLink

display(FileLink(output_zip))
